# 4. Модели: Обучение и Подбор Гиперпараметров
В этом ноутбуке мы обучим базовые модели (Decision Tree, Random Forest), градиентные бустинги (CatBoost, XGBoost) и ансамбль (StackingRegressor), используя логарифм цены (через `TransformedTargetRegressor`) и подберем гиперпараметры с помощью `GridSearchCV`.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from src.features import CraigslistFeatureEngineer

In [2]:
# Загрузка данных
data_path = Path("../data/interim/train_filtered.csv")
df = pd.read_csv(data_path, index_col=0)

# Небольшая выборка для ускорения обучения в академических целях
df = df.sample(3000, random_state=42)

y = df["price"].copy()
X = df.drop(columns=["price"]).copy()

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (3000, 18), y shape: (3000,)


In [3]:
# Вспомогательные функции для пайплайна
def categorical_without_description(X_df: pd.DataFrame) -> list[str]:
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    return [col for col in cat_cols if col != "description"]

def numeric_columns(X_df: pd.DataFrame) -> list[str]:
    return X_df.select_dtypes(exclude=["object", "category"]).columns.tolist()

def make_base_pipeline():
    feature_encoder = ColumnTransformer(
        transformers=[
            ("description_tfidf", TfidfVectorizer(max_features=500, ngram_range=(1, 1), min_df=10), "description"),
            ("categorical_ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01), categorical_without_description),
            ("numeric", "passthrough", numeric_columns),
        ],
        remainder="drop",
    )
    return Pipeline(steps=[("preprocess", CraigslistFeatureEngineer()), ("encode", feature_encoder)])

In [4]:
# Подготовка CV и метрики
cv = KFold(n_splits=3, shuffle=True, random_state=42) # 3 фолда
scoring = {"mae": make_scorer(mean_absolute_error, greater_is_better=False)}

In [5]:
# 1. Decision Tree (через TransformedTargetRegressor для логарифма цены)
dt_base = DecisionTreeRegressor(random_state=42)
dt_model = TransformedTargetRegressor(regressor=dt_base, func=np.log1p, inverse_func=np.expm1)

pipe_dt = Pipeline([("prep", make_base_pipeline()), ("model", dt_model)])

param_grid_dt = {"model__regressor__max_depth": [10, 20]}
gs_dt = GridSearchCV(pipe_dt, param_grid_dt, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)
gs_dt.fit(X, y)
print("Лучшие параметры DT:", gs_dt.best_params_)
print("Лучшая MAE DT:", -gs_dt.best_score_)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


Лучшие параметры DT: {'model__regressor__max_depth': 10}
Лучшая MAE DT: 6637.572698600329


In [6]:
# 2. Random Forest
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_model = TransformedTargetRegressor(regressor=rf_base, func=np.log1p, inverse_func=np.expm1)

pipe_rf = Pipeline([("prep", make_base_pipeline()), ("model", rf_model)])

param_grid_rf = {"model__regressor__n_estimators": [50], "model__regressor__max_depth": [10]}
gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)
gs_rf.fit(X, y)
print("Лучшие параметры RF:", gs_rf.best_params_)
print("Лучшая MAE RF:", -gs_rf.best_score_)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


Лучшие параметры RF: {'model__regressor__max_depth': 10, 'model__regressor__n_estimators': 50}
Лучшая MAE RF: 5408.077697754336


In [7]:
# 3. XGBoost
xgb_base = XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror')
xgb_model = TransformedTargetRegressor(regressor=xgb_base, func=np.log1p, inverse_func=np.expm1)

pipe_xgb = Pipeline([("prep", make_base_pipeline()), ("model", xgb_model)])

param_grid_xgb = {"model__regressor__n_estimators": [50], "model__regressor__max_depth": [6]}
gs_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)
gs_xgb.fit(X, y)
print("Лучшие параметры XGB:", gs_xgb.best_params_)
print("Лучшая MAE XGB:", -gs_xgb.best_score_)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


Лучшие параметры XGB: {'model__regressor__max_depth': 6, 'model__regressor__n_estimators': 50}
Лучшая MAE XGB: 5028.429524739583


In [8]:
# 4. CatBoost
cb_base = CatBoostRegressor(random_state=42, verbose=0, thread_count=-1)
cb_model = TransformedTargetRegressor(regressor=cb_base, func=np.log1p, inverse_func=np.expm1)

pipe_cb = Pipeline([("prep", make_base_pipeline()), ("model", cb_model)])

param_grid_cb = {"model__regressor__iterations": [100], "model__regressor__depth": [6]}
gs_cb = GridSearchCV(pipe_cb, param_grid_cb, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)
gs_cb.fit(X, y)
print("Лучшие параметры CatBoost:", gs_cb.best_params_)
print("Лучшая MAE CatBoost:", -gs_cb.best_score_)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


Лучшие параметры CatBoost: {'model__regressor__depth': 6, 'model__regressor__iterations': 100}
Лучшая MAE CatBoost: 4908.073816655141


In [9]:
# 5. Stacking (Ensemble)
# В качестве базовых моделей возьмем лучшие RF, XGBoost и CatBoost, а мета-моделью будет простая линейная регрессия (Ridge).
estimators = [
    ('rf', gs_rf.best_estimator_.named_steps['model'].regressor_),
    ('cb', gs_cb.best_estimator_.named_steps['model'].regressor_),
    ('xgb', gs_xgb.best_estimator_.named_steps['model'].regressor_)
]
stack_base = StackingRegressor(estimators=estimators, final_estimator=Ridge())
stack_model = TransformedTargetRegressor(regressor=stack_base, func=np.log1p, inverse_func=np.expm1)

pipe_stack = Pipeline([("prep", make_base_pipeline()), ("model", stack_model)])

# Обучаем ансамбль (без гридсерча, просто CV оценка)
stack_scores = cross_val_score(pipe_stack, X, y, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1)
stack_mae = -stack_scores.mean()

print(f"MAE Stacking Ensemble: {stack_mae:.2f}")

C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


MAE Stacking Ensemble: 4745.01


In [10]:
# Сравнение и сохранение
results = {
    "Decision Tree": -gs_dt.best_score_,
    "Random Forest": -gs_rf.best_score_,
    "XGBoost": -gs_xgb.best_score_,
    "CatBoost": -gs_cb.best_score_,
    "Stacking": stack_mae
}

results_df = pd.DataFrame(list(results.items()), columns=["Model", "MAE"]).sort_values("MAE")
print("Итоговые результаты:")
print(results_df)

best_model_name = results_df.iloc[0]["Model"]
print(f"Лучшая модель: {best_model_name}")

if best_model_name == "CatBoost":
    best_pipe = gs_cb.best_estimator_
elif best_model_name == "Stacking":
    best_pipe = pipe_stack.fit(X, y)
elif best_model_name == "XGBoost":
    best_pipe = gs_xgb.best_estimator_
elif best_model_name == "Random Forest":
    best_pipe = gs_rf.best_estimator_
else:
    best_pipe = gs_dt.best_estimator_

joblib.dump(best_pipe, "../data/interim/best_model_pipeline.pkl")
print("Финальная модель сохранена.")


Итоговые результаты:
           Model          MAE
4       Stacking  4745.005236
3       CatBoost  4908.073817
2        XGBoost  5028.429525
1  Random Forest  5408.077698
0  Decision Tree  6637.572699
Лучшая модель: Stacking


C:\Users\Alex\AppData\Local\Temp\ipykernel_6404\289616050.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()


Финальная модель сохранена.
